# ScholarRAG: M3, hybrid retrieval + reranking

**Goal:** beat BM25 on Hit@1 and MRR on set A (LLM questions) without losing dense's lead on set B1 (handwritten questions).

| Step | Setting |
|---|---|
| Candidates | Dense (`bge-small-en-v1.5`) and BM25, top 100 each, same indexes as the baseline |
| Fusion | Reciprocal Rank Fusion: `score(d) = Σ 1 / (60 + rank(d))` over the two lists |
| Rerank | Cross-encoder rescoring of the top 20 fused chunks; ranks 21+ keep their fused order |
| Rerankers | `cross-encoder/ms-marco-MiniLM-L-6-v2` (22M parameters, fast) and `BAAI/bge-reranker-base` (278M) |
| Chunk sizes | 350 words (default) and 250 words (chunk-size experiment: fits the reranker's 512-token limit better) |
| Questions | Set A (64) and set B1 (15), reported per source |

A reranker can only reorder the top 20, so the fused Hit@20 is the ceiling for every reranked score.

First run downloads the two reranker models (about 90 MB and 1.1 GB).

## 0. Setup

In [1]:
import re, json, time, unicodedata
import numpy as np, pandas as pd
from pathlib import Path

BASE = Path.cwd() / "data"   # jalankan dari root proyek
SIZES = [350, 250]
K_CAND = 100      # kandidat per retriever untuk fusion
K_RERANK = 20     # berapa teratas yang di-rerank
RRF_K = 60

qa = pd.read_csv(BASE / "eval" / "questions_v1.csv", dtype={"paper_id": str})
qa = qa[["id", "question", "evidence", "paper_id"]].assign(source="llm")
hw = pd.read_csv(BASE / "eval" / "questions_handwritten_v1.csv", dtype=str).fillna("")
hb = hw[hw.answerable.str.upper() == "TRUE"][["id", "question", "evidence", "paper_id"]].assign(source="handwritten")
qs = pd.concat([qa, hb], ignore_index=True)

chunk_sets = {s: pd.read_json(BASE / f"chunks_{s}.jsonl", lines=True, dtype={"paper_id": str}) for s in SIZES}
print(qs.source.value_counts().to_dict(), {s: len(c) for s, c in chunk_sets.items()})

{'llm': 64, 'handwritten': 15} {350: 3910, 250: 5945}


## 1. Gold chunks from evidence quotes

Same rule as the main notebook (section 7.5) and the chunk-size experiment.

In [2]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"-\s+(\w)", r"\1", s)
    return re.sub(r"\s+", " ", s).strip().lower()

def gold_for(row, ch):
    g = set()
    pids = [p.strip() for p in row.paper_id.split(";") if p.strip()]
    evs = [e.strip() for e in row.evidence.split(";") if e.strip()] if row.source == "handwritten" else [row.evidence]
    for pid, ev in zip(pids, evs):
        m = (ch.paper_id == pid) & ch.norm.str.contains(norm(ev), regex=False)
        g |= set(ch.index[m])
    return g

golds = {}
for size, ch in chunk_sets.items():
    ch["norm"] = ch.text.map(norm)
    golds[size] = {r.id: gold_for(r, ch) for _, r in qs.iterrows()}
    print(f"{size}: soal tanpa gold = {[i for i, g in golds[size].items() if not g] or 'tidak ada'}")

qs["paper_id"] = qs.paper_id.str.split(";").str[0]   # PaperHit memakai paper pertama

350: soal tanpa gold = tidak ada


250: soal tanpa gold = tidak ada


## 2. Dense and BM25 candidates

Dense embeddings come from the cache written by the earlier notebooks; they are recomputed only if the chunk ids changed.

In [3]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
QP = "Represent this sentence for searching relevant passages: "

def tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in ENGLISH_STOP_WORDS]

def load_or_embed(size, ch):
    ids = ch.chunk_id.tolist()
    emb_p, ids_p = (BASE / "index" / "emb_bge_small.npy", BASE / "index" / "chunk_ids.json") if size == 350 else \
                   (BASE / "index" / f"emb_bge_small_{size}.npy", BASE / "index" / f"chunk_ids_{size}.json")
    if emb_p.exists() and ids_p.exists() and json.load(open(ids_p)) == ids:
        return np.load(emb_p)
    E = emb_model.encode(ch.text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(emb_p, E); json.dump(ids, open(ids_p, "w"))
    return E

Q = emb_model.encode([QP + q for q in qs.question], normalize_embeddings=True)
rankings = {}   # rankings[(size, metode)][qid] = list indeks chunk, peringkat 1 dulu
for size, ch in chunk_sets.items():
    E = load_or_embed(size, ch)
    bm25 = BM25Okapi([tok(t) for t in ch.text])
    dense, sparse = {}, {}
    for j, q in enumerate(qs.itertuples()):
        dense[q.id] = np.argsort(-(E @ Q[j]))[:K_CAND].tolist()
        sparse[q.id] = np.argsort(-bm25.get_scores(tok(q.question)))[:K_CAND].tolist()
    rankings[(size, "Dense")], rankings[(size, "BM25")] = dense, sparse
    print(size, "siap:", E.shape)

W1019 16:37:47.615000 19964 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

350 siap: (3910, 384)


250 siap: (5945, 384)


## 3. Hybrid: Reciprocal Rank Fusion

In [4]:
def rrf(*lists, k=RRF_K):
    score = {}
    for lst in lists:
        for rank, idx in enumerate(lst, start=1):
            score[idx] = score.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

for size in SIZES:
    d, b = rankings[(size, "Dense")], rankings[(size, "BM25")]
    rankings[(size, "Hybrid (RRF)")] = {qid: rrf(d[qid], b[qid]) for qid in d}

## 4. Cross-encoder reranking

Each reranker scores (question, chunk) pairs for the top 20 fused chunks. Both models read at most 512 tokens per pair, so the share of pairs longer than that is reported per chunk size. Time is the average per question (20 pairs) on this machine.

In [5]:
from sentence_transformers import CrossEncoder

RERANKERS = {
    "Rerank MiniLM": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "Rerank bge-base": "BAAI/bge-reranker-base",
}

rerank_info = []
for name, model_id in RERANKERS.items():
    ce = CrossEncoder(model_id, max_length=512)
    for size, ch in chunk_sets.items():
        fused = rankings[(size, "Hybrid (RRF)")]
        out, n_long, n_pairs, t0 = {}, 0, 0, time.perf_counter()
        for q in qs.itertuples():
            top = fused[q.id][:K_RERANK]
            pairs = [(q.question, ch.text.iat[i]) for i in top]
            scores = ce.predict(pairs, batch_size=K_RERANK, show_progress_bar=False)
            out[q.id] = [top[i] for i in np.argsort(-scores)] + fused[q.id][K_RERANK:]
            n_long += sum(len(ce.tokenizer(a, b)["input_ids"]) > 512 for a, b in pairs)
            n_pairs += len(pairs)
        secs = (time.perf_counter() - t0) / len(qs)
        rankings[(size, name)] = out
        rerank_info.append({"reranker": name, "size": size, "pairs over 512 tokens": f"{n_long / n_pairs:.1%}",
                            "seconds per question": round(secs, 2)})
    del ce
pd.DataFrame(rerank_info).set_index(["reranker", "size"])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (633 > 512). Running this sequence through the model will result in indexing errors


pairs over 512 tokens  seconds per question
reranker        size                                            
Rerank MiniLM   350                  47.1%                  0.59
                250                   0.5%                  0.61
Rerank bge-base 350                  86.0%                  3.81
                250                   3.9%                  5.73

## 5. Results

In [6]:
METHODS = ["Dense", "BM25", "Hybrid (RRF)", "Rerank MiniLM", "Rerank bge-base"]

def ranks_for(size, method):
    ch, gold, rk = chunk_sets[size], golds[size], rankings[(size, method)]
    rows = []
    for q in qs.itertuples():
        ranked = rk[q.id]
        r_chunk = next((i + 1 for i, c in enumerate(ranked) if c in gold[q.id]), np.nan)
        papers = ch.paper_id.values[ranked[:10]]
        r_paper = next((i + 1 for i, p in enumerate(papers) if p == q.paper_id), np.nan)
        rows.append({"id": q.id, "source": q.source, "size": size, "method": method,
                     "rank_chunk": r_chunk, "rank_paper": r_paper})
    return rows

ranks = pd.DataFrame([r for s in SIZES for m in METHODS for r in ranks_for(s, m)])
ranks.to_csv(BASE / "eval" / "results_m3_ranks.csv", index=False)

def metrics(d):
    return pd.Series({"n": len(d), "Hit@1": (d.rank_chunk <= 1).mean(), "Hit@5": (d.rank_chunk <= 5).mean(),
                      "Hit@10": (d.rank_chunk <= 10).mean(), "Hit@20": (d.rank_chunk <= 20).mean(),
                      "MRR": (1 / d.rank_chunk).fillna(0).mean(), "PaperHit@10": (d.rank_paper <= 10).mean()})

results = (ranks.groupby(["source", "size", "method"])[["rank_chunk", "rank_paper"]]
           .apply(metrics)
           .reindex(pd.MultiIndex.from_product([["llm", "handwritten"], SIZES, METHODS],
                                               names=["source", "size", "method"])))
results.to_csv(BASE / "eval" / "results_m3.csv")
results.round(3)

n  Hit@1  Hit@5  Hit@10  Hit@20    MRR  \
source      size method                                                       
llm         350  Dense            64.0  0.391  0.656   0.797   0.828  0.514   
                 BM25             64.0  0.625  0.906   0.938   0.969  0.752   
                 Hybrid (RRF)     64.0  0.516  0.844   0.875   0.906  0.656   
                 Rerank MiniLM    64.0  0.562  0.859   0.906   0.906  0.689   
                 Rerank bge-base  64.0  0.656  0.875   0.906   0.906  0.743   
            250  Dense            64.0  0.359  0.641   0.688   0.781  0.478   
                 BM25             64.0  0.578  0.891   0.906   0.969  0.708   
                 Hybrid (RRF)     64.0  0.469  0.781   0.828   0.922  0.601   
                 Rerank MiniLM    64.0  0.516  0.797   0.891   0.922  0.638   
                 Rerank bge-base  64.0  0.641  0.891   0.922   0.922  0.739   
handwritten 350  Dense            15.0  0.600  0.733   0.800   0.867  0.668   
                 BM25             15.0  0.200  0.800   0.933   0.933  0.443   
                 Hybrid (RRF)     15.0  0.533  0.800   0.933   0.933  0.644   
                 Rerank MiniLM    15.0  0.400  0.667   0.867   0.933  0.510   
                 Rerank bge-base  15.0  0.733  0.933   0.933   0.933  0.786   
            250  Dense            15.0  0.533  0.733   0.800   0.800  0.629   
                 BM25             15.0  0.333  0.867   0.867   0.933  0.508   
                 Hybrid (RRF)     15.0  0.600  0.733   0.933   0.933  0.680   
                 Rerank MiniLM    15.0  0.200  0.800   0.933   0.933  0.440   
                 Rerank bge-base  15.0  0.667  0.867   0.933   0.933  0.727   

                                  PaperHit@10  
source      size method                        
llm         350  Dense                  0.922  
                 BM25                   1.000  
                 Hybrid (RRF)           1.000  
                 Rerank MiniLM          1.000  
                 Rerank bge-base        1.000  
            250  Dense                  0.891  
                 BM25                   1.000  
                 Hybrid (RRF)           0.984  
                 Rerank MiniLM          0.969  
                 Rerank bge-base        1.000  
handwritten 350  Dense                  1.000  
                 BM25                   1.000  
                 Hybrid (RRF)           1.000  
                 Rerank MiniLM          1.000  
                 Rerank bge-base        1.000  
            250  Dense                  1.000  
                 BM25                   1.000  
                 Hybrid (RRF)           1.000  
                 Rerank MiniLM          1.000  
                 Rerank bge-base        1.000

### 5.1 Question by question against BM25

Averages over 15 questions hide a lot, so this counts, per question, whether each method puts the first gold chunk higher (better), at the same rank (same) or lower (worse) than BM25 does. Missing from the top 100 counts as rank 101.

In [7]:
base = ranks[ranks.method == "BM25"].set_index(["size", "id"]).rank_chunk.fillna(101)
rows = []
for (size, method, source), d in ranks[ranks.method != "BM25"].groupby(["size", "method", "source"]):
    r = d.set_index(["size", "id"]).rank_chunk.fillna(101)
    b = base.loc[r.index]
    rows.append({"source": source, "size": size, "method": method,
                 "better": int((r < b).sum()), "same": int((r == b).sum()), "worse": int((r > b).sum())})
(pd.DataFrame(rows).set_index(["source", "size", "method"])
   .reindex(pd.MultiIndex.from_product([["llm", "handwritten"], SIZES, METHODS[:1] + METHODS[2:]],
                                       names=["source", "size", "method"])))

better  same  worse
source      size method                              
llm         350  Dense                 9    20     35
                 Hybrid (RRF)          9    32     23
                 Rerank MiniLM        11    31     22
                 Rerank bge-base      16    31     17
            250  Dense                 6    22     36
                 Hybrid (RRF)         10    33     21
                 Rerank MiniLM        14    26     24
                 Rerank bge-base      17    34     13
handwritten 350  Dense                 9     2      4
                 Hybrid (RRF)          9     2      4
                 Rerank MiniLM         5     3      7
                 Rerank bge-base       8     6      1
            250  Dense                 7     4      4
                 Hybrid (RRF)          7     6      2
                 Rerank MiniLM         4     5      6
                 Rerank bge-base       6     6      3

## 6. Reading the results

Dense and BM25 match the earlier notebooks (Hit@k identical; MRR is 0.002 higher here because ranks 21–100 now count).

**Main comparison, 350-word chunks:**

| Method | Set A Hit@1 | Set A MRR | Set B1 Hit@1 | Set B1 MRR | Seconds / question |
|---|---|---|---|---|---|
| BM25 | 0.625 | **0.752** | 0.200 | 0.443 | – |
| Dense | 0.391 | 0.514 | 0.600 | 0.668 | – |
| Hybrid (RRF) | 0.516 | 0.656 | 0.533 | 0.644 | – |
| Hybrid + MiniLM | 0.562 | 0.689 | 0.400 | 0.510 | 0.6 |
| **Hybrid + bge-reranker-base** | **0.656** | 0.743 | **0.733** | **0.786** | 3.8 |

**1. Hybrid + `bge-reranker-base` is the best overall method.** On set A it ties BM25 (Hit@1 +2 questions, MRR −0.009; question by question: 16 better, 31 same, 17 worse). On set B1 it is clearly best: Hit@1 0.733 vs 0.600 (dense) and 0.200 (BM25), and better than BM25 on 8 of 15 questions, worse on 1. **The M3 target is partly met:** Hit@1 on set A is above BM25, MRR is level, and the lead on handwritten questions grows.

**2. RRF alone hurts on set A** (MRR 0.752 → 0.656). Fusing with the weaker dense list pushes some BM25 hits down. Four set A questions are in BM25's top 20 (one at rank 1) but absent from dense's top 100; RRF gives them only half a score and drops them to ranks 46–50, outside what the reranker sees. That is why the fused Hit@20 (0.906) is below BM25's (0.969), and it caps every reranked score.

**3. MiniLM does not help.** It is trained on web search queries (MS MARCO) and makes handwritten questions worse (Hit@1 0.533 → 0.400). `bge-reranker-base` is the one to keep.

**4. 350-word chunks stay the default, even for the reranker.** With bge-reranker-base, 86% of (question, chunk) pairs at 350 words are longer than 512 tokens (its tokenizer splits text into more pieces than BERT's), against 3.9% at 250 words. Yet 350 scores the same or better on both sets, so the cut-off tail of a chunk rarely held what the reranker needed.

**5. Cost:** about 4 seconds per question for 20 pairs on an M1 laptop. Acceptable for evaluation; for an interactive demo (M6) it would need fewer pairs, half precision or a GPU.

### Decision and next step

- **Pipeline for M4:** 350-word chunks → dense + BM25 → RRF → `bge-reranker-base` on the top 20.
- **Fix the candidate pool before M4 (small change):** rerank the union of BM25's top 20 and dense's top 20 instead of RRF's top 20, so a chunk that only one retriever finds still reaches the reranker. This would recover the four lost set A questions, at about twice the reranking cost.

## 7. Candidate-pool fix: rerank the union of both retrievers

Section 6 showed that RRF drops four set A questions that only BM25 finds, so the reranker never sees them. Here the reranker scores the **union of BM25's top 20 and dense's top 20** (duplicates removed) instead of RRF's top 20. Chunks outside the pool keep their RRF order.

The same scores also give a cheaper variant for free: the union of each retriever's **top 10**, which costs about as much as the current setup (≈ 20 pairs).

Only `bge-reranker-base` and 350-word chunks are tested (the winners of section 6). The current M3 result is read from `eval/results_m3_ranks.csv`, so this section can run without re-running section 4.

In [5]:
from sentence_transformers import CrossEncoder

SIZE = 350
ch = chunk_sets[SIZE]
d_rk, b_rk, f_rk = (rankings[(SIZE, m)] for m in ["Dense", "BM25", "Hybrid (RRF)"])
ce = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

def dedup(xs):
    return list(dict.fromkeys(xs))

pool_rank, pool_info = {"Union 10+10 + bge-base": {}, "Union 20+20 + bge-base": {}}, []
t0 = time.perf_counter()
for q in qs.itertuples():
    pool20 = dedup(b_rk[q.id][:20] + d_rk[q.id][:20])
    s = dict(zip(pool20, ce.predict([(q.question, ch.text.iat[i]) for i in pool20],
                                    batch_size=16, show_progress_bar=False)))
    pool10 = dedup(b_rk[q.id][:10] + d_rk[q.id][:10])     # subset dari pool20, skor sudah ada
    for name, pool in [("Union 10+10 + bge-base", pool10), ("Union 20+20 + bge-base", pool20)]:
        head = sorted(pool, key=s.get, reverse=True)
        ps = set(pool)
        pool_rank[name][q.id] = head + [i for i in f_rk[q.id] if i not in ps]
    pool_info.append({"id": q.id, "pool10": len(pool10), "pool20": len(pool20)})
secs20 = (time.perf_counter() - t0) / len(qs)
del ce

pool_info = pd.DataFrame(pool_info)
print(f"rata-rata pasangan: union 10+10 = {pool_info.pool10.mean():.1f}, union 20+20 = {pool_info.pool20.mean():.1f}")
print(f"waktu union 20+20: {secs20:.2f} detik per soal")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

rata-rata pasangan: union 10+10 = 15.7, union 20+20 = 32.7
waktu union 20+20: 6.36 detik per soal


In [6]:
def ranks_from(rk, method):
    gold = golds[SIZE]
    rows = []
    for q in qs.itertuples():
        ranked = rk[q.id]
        r_chunk = next((i + 1 for i, c in enumerate(ranked) if c in gold[q.id]), np.nan)
        papers = ch.paper_id.values[ranked[:10]]
        r_paper = next((i + 1 for i, p in enumerate(papers) if p == q.paper_id), np.nan)
        rows.append({"id": q.id, "source": q.source, "method": method, "rank_chunk": r_chunk, "rank_paper": r_paper})
    return pd.DataFrame(rows)

prev = pd.read_csv(BASE / "eval" / "results_m3_ranks.csv")
prev = prev[(prev["size"] == SIZE) & prev.method.isin(["BM25", "Rerank bge-base"])].drop(columns="size")
prev["method"] = prev.method.replace({"Rerank bge-base": "RRF top 20 + bge-base (M3)"})
new = pd.concat([ranks_from(rk, m) for m, rk in pool_rank.items()])
pool_ranks = pd.concat([prev, new], ignore_index=True)
pool_ranks.to_csv(BASE / "eval" / "results_m3_pool_ranks.csv", index=False)

# berapa soal yang gold-nya masuk kandidat yang di-rerank (batas atas skor rerank)
gold = golds[SIZE]
in_pool = {
    "RRF top 20 + bge-base (M3)": {q: bool(gold[q] & set(f_rk[q][:20])) for q in qs.id},
    "Union 10+10 + bge-base": {q: bool(gold[q] & set(dedup(b_rk[q][:10] + d_rk[q][:10]))) for q in qs.id},
    "Union 20+20 + bge-base": {q: bool(gold[q] & set(dedup(b_rk[q][:20] + d_rk[q][:20]))) for q in qs.id},
}
pairs = {"RRF top 20 + bge-base (M3)": 20.0, "Union 10+10 + bge-base": pool_info.pool10.mean(),
         "Union 20+20 + bge-base": pool_info.pool20.mean()}

METHODS_P = ["BM25", "RRF top 20 + bge-base (M3)", "Union 10+10 + bge-base", "Union 20+20 + bge-base"]
rows = []
for (src, m), d in pool_ranks.groupby(["source", "method"]):
    rows.append({"source": src, "method": m, "n": len(d), "pairs / question": pairs.get(m, np.nan),
                 "gold in pool": np.mean([in_pool[m][i] for i in d.id]) if m in in_pool else np.nan,
                 "Hit@1": (d.rank_chunk <= 1).mean(), "Hit@5": (d.rank_chunk <= 5).mean(),
                 "Hit@10": (d.rank_chunk <= 10).mean(), "MRR": (1 / d.rank_chunk).fillna(0).mean(),
                 "PaperHit@10": (d.rank_paper <= 10).mean()})
pool_results = (pd.DataFrame(rows).set_index(["source", "method"])
                .reindex(pd.MultiIndex.from_product([["llm", "handwritten"], METHODS_P], names=["source", "method"])))
pool_results.to_csv(BASE / "eval" / "results_m3_pool.csv")
pool_results.round(3)

n  pairs / question  gold in pool  \
source      method                                                           
llm         BM25                        64               NaN           NaN   
            RRF top 20 + bge-base (M3)  64            20.000         0.906   
            Union 10+10 + bge-base      64            15.722         0.938   
            Union 20+20 + bge-base      64            32.671         0.969   
handwritten BM25                        15               NaN           NaN   
            RRF top 20 + bge-base (M3)  15            20.000         0.933   
            Union 10+10 + bge-base      15            15.722         0.933   
            Union 20+20 + bge-base      15            32.671         0.933   

                                        Hit@1  Hit@5  Hit@10    MRR  \
source      method                                                    
llm         BM25                        0.625  0.906   0.938  0.752   
            RRF top 20 + bge-base (M3)  0.656  0.875   0.906  0.743   
            Union 10+10 + bge-base      0.625  0.906   0.938  0.740   
            Union 20+20 + bge-base      0.625  0.891   0.953  0.742   
handwritten BM25                        0.200  0.800   0.933  0.443   
            RRF top 20 + bge-base (M3)  0.733  0.933   0.933  0.786   
            Union 10+10 + bge-base      0.733  0.933   0.933  0.790   
            Union 20+20 + bge-base      0.733  0.867   0.933  0.781   

                                        PaperHit@10  
source      method                                   
llm         BM25                                1.0  
            RRF top 20 + bge-base (M3)          1.0  
            Union 10+10 + bge-base              1.0  
            Union 20+20 + bge-base              1.0  
handwritten BM25                                1.0  
            RRF top 20 + bge-base (M3)          1.0  
            Union 10+10 + bge-base              1.0  
            Union 20+20 + bge-base              1.0

### 7.1 Question by question

Each union variant against BM25 and against the current M3 setup (better / same / worse rank of the first gold chunk; outside the top 100 counts as 101).

In [7]:
r = pool_ranks.assign(rank_chunk=pool_ranks.rank_chunk.fillna(101)).pivot(index="id", columns="method", values="rank_chunk")
src = pool_ranks.drop_duplicates("id").set_index("id").source
rows = []
for m in METHODS_P[2:]:
    for ref in ["BM25", "RRF top 20 + bge-base (M3)"]:
        for s in ["llm", "handwritten"]:
            ids = src[src == s].index
            a, b = r.loc[ids, m], r.loc[ids, ref]
            rows.append({"method": m, "vs": ref, "source": s,
                         "better": int((a < b).sum()), "same": int((a == b).sum()), "worse": int((a > b).sum())})
pd.DataFrame(rows).set_index(["method", "vs", "source"])

better  same  \
method                 vs                         source                      
Union 10+10 + bge-base BM25                       llm              15    32   
                                                  handwritten       9     5   
                       RRF top 20 + bge-base (M3) llm               7    54   
                                                  handwritten       1    14   
Union 20+20 + bge-base BM25                       llm              15    33   
                                                  handwritten       8     5   
                       RRF top 20 + bge-base (M3) llm               5    55   
                                                  handwritten       0    14   

                                                               worse  
method                 vs                         source              
Union 10+10 + bge-base BM25                       llm             17  
                                                  handwritten      1  
                       RRF top 20 + bge-base (M3) llm              3  
                                                  handwritten      0  
Union 20+20 + bge-base BM25                       llm             16  
                                                  handwritten      2  
                       RRF top 20 + bge-base (M3) llm              4  
                                                  handwritten      1

### 7.2 The four questions RRF dropped

In [8]:
r.loc[["q006", "q035", "q044", "q054"], METHODS_P].astype(int)

method,BM25,RRF top 20 + bge-base (M3),Union 10+10 + bge-base,Union 20+20 + bge-base
id,,,,
q006,1,46,2,2
q035,19,47,47,21
q044,6,50,6,6
q054,17,47,47,3


## 8. Reading the candidate-pool results

| Set A (n = 64) | Pairs / question | Gold in pool | Hit@1 | Hit@5 | Hit@10 | MRR |
|---|---|---|---|---|---|---|
| BM25 | – | – | 0.625 | 0.906 | 0.938 | **0.752** |
| RRF top 20 + bge-base (M3) | 20 | 0.906 | **0.656** | 0.875 | 0.906 | 0.743 |
| **Union 10+10 + bge-base** | 15.7 | 0.938 | 0.625 | **0.906** | 0.938 | 0.740 |
| Union 20+20 + bge-base | 32.7 | **0.969** | 0.625 | 0.891 | **0.953** | 0.742 |

On set B1 (n = 15) all three reranked variants score Hit@1 0.733 and MRR 0.78–0.79, far above BM25 (0.200 / 0.443).

**1. The fix works for recall.** The share of questions whose gold chunk reaches the reranker rises from 0.906 to 0.938 (10+10) and 0.969 (20+20, equal to BM25's Hit@20). Three of the four dropped questions come back into the top 10 (q006 at rank 2, q044 at 6, q054 at 3 with 20+20); q035 stays outside.

**2. It does not raise Hit@1.** A bigger pool also gives the reranker more near-miss chunks to confuse with the right one. Against the M3 setup, union 10+10 is better on 7 set A questions and worse on 3, two of which lose rank 1; union 20+20 is better on 5 and worse on 4. On set A every reranked variant stays within about two questions of BM25 at every cutoff, which is inside the noise of a 64-question set.

**3. Union 10+10 is the best trade-off.** It is cheaper than M3 (15.7 pairs instead of 20, roughly 3 s per question on an M1, estimated from the 20+20 timing), it matches BM25 on set A at Hit@5 and Hit@10, and it keeps the large lead on handwritten questions. Union 20+20 doubles the cost (6.4 s per question) for no gain.

**4. What matters next is Hit@5, not Hit@1.** In M4 the LLM reads the top few chunks, so the right chunk only needs to be among them. Union 10+10 has the best Hit@5 on set A (0.906, tied with BM25) and ties for best on set B1 (0.933).

### Decision

**Retrieval for M4:** 350-word chunks → BM25 top 10 ∪ dense top 10 → `bge-reranker-base` → top 5 chunks to the LLM. M3 is closed: on LLM-written questions the pipeline matches BM25, and on paraphrased questions it is far ahead (Hit@1 0.733 vs 0.200).